### Ingesting drivers file

In [0]:
%run ../00-common/1.environment_config

In [0]:
%run ../00-common/2.bronze_helpers

In [0]:
dbutils.widgets.text("p_batch_id", "")  
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
source_file = f'{landing_folder_path}/{v_batch_id}/drivers.json'
table_name = f'{catalog_name}.{bronze_schema}.drivers'

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DateType

name_schema = StructType([
    StructField('familyName', StringType(), True),
    StructField('givenName', StringType(), True)
])

drivers_schema = StructType([
    StructField('driverId', StringType(), True),
    StructField('dateOfBirth', DateType(), True),
    StructField('name', name_schema),
    StructField('nationality', StringType(), True),
    StructField('url', StringType(), True)
])



In [0]:
drivers_df = (
    spark.read
    .format('json')
    .option('mode','FAILFAST')
    .schema(drivers_schema) 
    .load(source_file)
)

In [0]:
display(drivers_df)

In [0]:
# Ingesting metadata
drivers_df_final = add_file_metadata(drivers_df)

In [0]:
display(drivers_df_final)

In [0]:
write_to_bronze(drivers_df_final, table_name, v_batch_id)

In [0]:
display(spark.table(table_name))